In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from pathlib import Path

load_dotenv()

client = Anthropic(
    default_headers={
        "anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"
    }
)
model = "claude-haiku-5-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_effort="medium",
):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {"type": "adaptive"}
        params["output_config"] = {"effort": thinking_effort}

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


def upload(file_path):
    path = Path(file_path)
    extension = path.suffix.lower()

    mime_type_map = {
        ".pdf": "application/pdf",
        ".txt": "text/plain",
        ".md": "text/plain",
        ".py": "text/plain",
        ".js": "text/plain",
        ".html": "text/plain",
        ".css": "text/plain",
        ".csv": "text/csv",
        ".json": "application/json",
        ".xml": "application/xml",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".xls": "application/vnd.ms-excel",
        ".jpeg": "image/jpeg",
        ".jpg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }

    mime_type = mime_type_map.get(extension)

    if not mime_type:
        raise ValueError(f"Unknown mimetype for extension: {extension}")
    filename = path.name

    with open(file_path, "rb") as file:
        return client.beta.files.upload(file=(filename, file, mime_type))


def list_files():
    return client.beta.files.list()


def delete_file(id):
    return client.beta.files.delete(id)


def download_file(id, filename=None):
    file_content = client.beta.files.download(id)

    if not filename:
        file_metadata = get_metadata(id)
        file_content.write_to_file(file_metadata.filename)
    else:
        file_content.write_to_file(filename)


def generated_file_ids(message):
    file_ids = []
    for block in message.content:
        if block.type != "bash_code_execution_tool_result":
            continue
        result = block.content
        if result.type == "bash_code_execution_result":
            file_ids.extend(output.file_id for output in result.content)
    return list(dict.fromkeys(file_ids))


def get_metadata(id):
    return client.beta.files.retrieve_metadata(id)

In [3]:
file_metadata = upload("streaming.csv")
file_metadata

BetaFileMetadata(id='file_013juAaKQx89brBu6e6TgYrV', created_at=datetime.datetime(2026, 10, 9, 2, 6, 23, 601243, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False, expires_at=None, scope=None)

In [4]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
Run a detailed analysis to determine major drivers of churn.
Your final output should include at least one detailed plot summarizing your findings.
Save every plot in the directory given by the OUTPUT_DIR environment variable so it is returned as a downloadable file.

Critical note: Every time you execute code, you're starting with a completely clean slate. 
No variables or library imports from previous executions exist. You need to redeclare/reimport all variables/libraries.
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

response = chat(messages, tools=[{"type": "code_execution_20250825", "name": "code_execution"}])
response

Message(id='msg_011CfqqzWMawrwZyTgFeovom', container=Container(id='container_01NPtk6Kw4V9AXHXK1Q1MiFj', expires_at=datetime.datetime(2026, 10, 9, 3, 7, 42, 463136, tzinfo=TzInfo(0)), skills=None), content=[ThinkingBlock(signature='CAQS7AQKEAgSGAI4AUIIdGhpbmtpbmcSDGM8U6MK3voimlTzmBoMEIOKYIa8X7/XQP5EIjBaXEEUvcwx4Qrwe6iCgJ5wh43qXxWjcScRf4xEHzqUy1cSDCn4D9u74dMUCJh1vHoqiQRfALUxmsLnd9yDU6Ze9LQMLtX4vzJQ8W2FIvHwMrlRCUO72dzwEcv/BZnXyNDayz5KXysK6gO8iDW1G+pu85KwxmM2XopzCIaFx7JRLFQUGMwD1Boia119ceXtFmlguHhw4R5Il6Tr/+MEdH2j458+ZW4s/F2Jt6rQJEV2xezaF5PUUzc0paYbG/oViy9UrarZsrSHO+vcDufyUuijiNdx6h3a52WmkomyoHd4bK7b3k+7agbM3efO721Xy88iao87ViUd4Ig6Yri6sQXgZuSRKKSdGD+qquVi/V05KXmh6kINMIaMH1P6ijrSMC2/zh8jr4zhz/ky/9cBj9c9t9y55Ob1PjO9drJ1sZr2j1yjHRu53H3oRPTl/eXbV1m7fo5UuOIFIPr0VKiWjeVQSw8zJyk5q33ENenfwzoM7tO/Nt4su7EEi2LJk80oAFElw0uNsuHxJYemz/JBppNA8YKiP1qsMKkyxd2iI6haHbv1CxrMRJzKhyHvUxF1LWtcGtEK1yLqCQBvNpD95UbxmrV1iSJ82/R0dKTrwErsFzXSu/au7To+/ddJeICKjRkzrkYFHVzMMt1L1ftiO3gsRpWs9v9hH5C3mxwJq9Nf1T/nyfHUwkcCgEUil

In [5]:
file_ids = generated_file_ids(response)
if not file_ids:
    raise ValueError("No generated files were returned. Inspect response.content and response.stop_reason, then rerun the analysis saving plots in OUTPUT_DIR.")

for file_id in file_ids:
    download_file(file_id)
    print(f"Downloaded generated file: {file_id}")

Downloaded generated file: file_012WPNKbq5Qf4kPa4Gi7s1Zs
Downloaded generated file: file_01MHVNSaUnSb3QnurZ7tjLH1
Downloaded generated file: file_01E5fpyLFp7EiBivR5XkJmAT
Downloaded generated file: file_01RwizRd6gw1XH9wpS1x1Pwg
